# EDA

## 1. Necessary Packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

## 2. Preliminary Setup

In [ ]:
# Visualization!
sns.set_theme(style = "whitegrid", context = "notebook")

# Decimals!
pd.set_option("display.float_format", "{:.3f}".format)

# Plot sizes!
FIG = (18, 16)
FS = 14

## 3. Inspection Of Datasets

In [ ]:
# Load clean datasets!
all = pd.read_csv("../data/clean/combo_clean_all.csv")
judge = pd.read_csv("../data/clean/combo_clean_judge.csv")

# All players!
print("First few rows of dataset with all players:\n")
print(all.head(), "\n")
print(f"Overall shape: {all.shape}\n")

# Judge only!
print("First few rows of dataset with only Judge:\n")
print(judge.head(), "\n")
print(f"Overall shape: {judge.shape}")

## 4. Checking Data Within Each Dataset

In [ ]:
# All players dataset basic info!
print("All players dataset basic info:\n")
print(all.info(), "\n")

# Missing values in all players dataset!
missing_counts = all.isna().sum()
print(missing_counts[missing_counts > 0], "\n")
print("Total missing values:", all.isna().sum().sum(), "\n")

# Summary statistics of all players dataset!
print("All players dataset summary stats:\n")
print(all.describe().transpose().round(3), "\n")

# Summary statistics of Judge dataset!
print("Judge dataset summary stats:\n")
print(judge.describe().transpose().round(3))

## 5. Plotting Distributions Of Some Power Metrics (Judge Highlighted By Season With Red-Dashed Lines)

In [ ]:
# Define key metrics to visualize!
dist_metrics = {"hr": "HR",
                "slg": "SLG",
                "barrel_percentage": "Barrel Percentage",
                "sweet_spot_percentage": "Sweet Spot Percentage",
                "hard_hit_percentage": "Hard Hit Percentage",
                "avg_exit_velocity": "Average EV (in mph)"}

# Plot each distribution!
plt.figure(figsize = FIG)
for idx, (col, label) in enumerate(dist_metrics.items(), 1):
    plt.subplot(3, 2, idx)
    sns.histplot(all[col], kde = True, color = "gray", alpha = 0.6)
    
    # Add Judge's values as red lines!
    for val in judge[col]:
        plt.axvline(val, color = "red", linestyle = "--", linewidth = 2)
    plt.title(f"Distribution of {label}", fontsize = FS)
    plt.xlabel(label)
    plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

Note:

- These distributions show that Aaron Judge *consistently performs well above league average* across all six power-related metrics examined: HR, SLG, barrel percentage, sweet spot percentage, hard hit percentage, and average EV. In each case, Judge's values lie on the right tail of the league distribution, underscoring not only his exceptional raw power but also the consistency with which he generates high-quality contact. These metrics capture different but related aspects of batted-ball performance: EV reflects raw strength, hard hit and barrel percentages capture how often he produces optimal contact, and SLG and HR represent the outcomes that result. Judge's placement in the upper percentiles across all of them highlights a complete power profile rather than dominance in just one dimension. This pattern makes intuitive sense given Judge's well-established hitting profile. His combination of *size, bat speed, and refined plate discipline* enables him to maximize both frequency and quality of impactful contact. While many hitters excel in one or two power metrics, very few sustain exceptional performance simultaneously across all six. Judge's consistent outperformance of league norms therefore reinforces what scouting assessments and prior performance already suggest: he is not merely a high-power hitter, but also one of the most potent and reliably productive in modern baseball.

## 6. Making Scatterplots That Compare Some More Metrics (Judge Again Highlighted By Season With Red Dots)

In [ ]:
# Set figure size!
plt.figure(figsize = FIG)

# Set pairs for comparison!
scatter_pairs = [("avg_exit_velocity", "hr", "Average EV (in mph) vs. HR"),
                 ("barrel_percentage", "weighted_runs_created_plus", "Barrel Percentage vs. wRC+"),
                 ("hard_hit_percentage", "slg", "Hard Hit Percentage vs. SLG"),
                 ("avg_launch_angle", "hr", "LA vs. HR")]

# Plot each pair!
for idx, (xcol, ycol, title) in enumerate(scatter_pairs, 1):
    plt.subplot(2, 2, idx)
    
    # League data points!
    sns.scatterplot(data = all,
                    x = xcol, 
                    y = ycol,
                    color = "gray", 
                    alpha = 0.4, 
                    s = 40)
    
    # Judge points!
    sns.scatterplot(data = judge,
                    x = xcol, 
                    y = ycol,
                    color = "red", 
                    s = 90, 
                    edgecolor = "black")
    plt.title(title, fontsize = FS)
    plt.xlabel(title.split(" vs. ")[0], fontsize = FS - 2)
    plt.ylabel(title.split(" vs. ")[1], fontsize = FS - 2)
plt.tight_layout()
plt.show()

Note: 

- Across these scatterplots, there is again clear evidence of Judge's exceptional offensive profile, as well as the degree to which various power and contact quality metrics correlate across the league. In the `Average EV (in mph) vs. HR` plot, Judge appears firmly in the upper-right quadrant: he consistently produces some of the highest EV in baseball while also hitting a remarkably large number of HR. The league-wide trend shows a positive correlation between these two variables; i.e., hit the ball harder, and one tends to hit it out more often, which makes Judge's placement *intuitive but nonetheless striking in its extremity*. A similar pattern emerges in the `Barrel Percentage vs. wRC+` plot, where Judge is again in a league of his own. He regularly posts barrel percentages above $25\%$ and multiple seasons with a wRC+ above $200$, both of which are extraordinary. This scatter also shows a positive correlation, reflecting the fact that frequent, optimally struck balls translate directly into overall offensive value.

- The two remaining plots reinforce this theme of Judge outperforming league norms while also illustrating how different underlying metrics relate to one another. In the `Hard Hit Percentage vs. SLG` visualization, Judge consistently appears among the top performers in both categories, though the league-wide relationship is *slightly less tightly clustered than in the previous examples*, suggesting that while hard contact strongly contributes to getting extra-base hits, there is more variation introduced by other variables. Finally, the `LA vs. HR` plot offers a subtle but important insight: Judge does not rely on extreme LA to generate power. Despite having LAs that are not among the highest in the league, he still produces exceptional HR totals, largely because his combination of strength and contact quality allows even moderately elevated balls to be HR. This divergence from the typical pattern underscores the uniqueness of Judge's power profile and helps explain why he consistently sits at the far right edge of so many offensive distributions.

## 7. Plotting Year-Over-Year Trends (Judge vs. League)

In [ ]:
# Compute league averages by season!
league_means = all.groupby("season").agg({"hr": "mean",
                                          "weighted_runs_created_plus": "mean",
                                          "barrel_percentage": "mean",
                                          "hard_hit_percentage": "mean",
                                          "avg_exit_velocity": "mean",
                                          "expected_weighted_on_base_average": "mean"}).reset_index()

# Metrics to plot!
trend_metrics = {"hr": "HR",
                 "weighted_runs_created_plus": "wRC+",
                 "barrel_percentage": "Barrel Percentage",
                 "hard_hit_percentage": "Hard Hit Percentage",
                 "avg_exit_velocity": "Average EV (in mph)",
                 "expected_weighted_on_base_average": "Expected wOBA"}

# Set figure size!
plt.figure(figsize = FIG)

# Plot each pair!
for idx, (col, label) in enumerate(trend_metrics.items(), 1):
    plt.subplot(3, 2, idx)

    # League trend!
    sns.lineplot(data = league_means,
                 x = "season",
                 y = col,
                 marker = "o",
                 color = "blue",
                 label = "League Average")

    # Judge trend!
    sns.lineplot(data = judge,
                 x = "season",
                 y = col,
                 marker = "o",
                 color = "red",
                 label = "Aaron Judge")
    plt.title(f"{label} Over Time", fontsize = FS)
    plt.xlabel("Season", fontsize = FS - 2)
    plt.ylabel(label, fontsize = FS - 2)
    plt.legend()
plt.tight_layout()
plt.show()

Note: These year-over-year plots illustrate that Judge has been a *significantly better hitter than the vast majority of the league since the moment he arrived in 2017*. What makes this even more noteworthy is that Judge was something of a "late bloomer" by modern MLB standards: his breakout rookie season came at age 25, whereas many exceptional prospects debut, and often establish themselves as stars, in their early twenties. Despite entering the league later than typical high-upside hitters, Judge immediately performed at an exceptional level and has sustained that excellence across multiple seasons. His year-to-year metrics not only remain consistently above league averages but frequently rank among the very best in baseball.

## 8. Plotting Correlation Heatmaps

For two variables $X_1$ and $X_2$, each with $n$ *observations*, the *Pearson correlation coefficient* is defined as

$$
\rho_{X_1,\,X_2}=\frac{\mathrm{Cov}(X_1,\,X_2)}{\sigma_{X_1}\cdot\sigma_{X_2}} \tag*{(1)}
$$

where $\sigma_{X_1},\,\sigma_{X_2}$ are the *standard deviations* for $X_1$ and $X_2$, respectively, and

$$
\mathrm{Cov}(X_1,\,X_2)=\frac{1}{n}\sum_{i=1}^n(x_{1i}-\bar{x}_1)(x_{2i}-\bar{x}_2) \tag*{(2)}
$$

is the *covariance* between $X_1$ and $X_2$ where $\bar{x}_1,\,\bar{x}_2$ are the *means* of $X_1$ and $X_2$.

### A. All Metrics

In [ ]:
# Identify numeric columns!
numeric_cols = all.select_dtypes(include = [np.number]).columns

# Columns to exclude from correlation analysis!
exclude = ["fangraphs_id", "mlbam_id", "season"]

# Keep only useful numeric columns!
corr_cols = [col for col in numeric_cols if col not in exclude]

# Compute correlation matrix!
corr = all[corr_cols].corr()

# Plot heatmap!
plt.figure(figsize = FIG)
sns.heatmap(corr,
            cmap = "coolwarm",
            center = 0,
            square = False,
            linewidths = 0.1)
plt.title("Correlation Heatmap For All Metrics", fontsize = FS + 8)
plt.tight_layout()
plt.show()

### B. Key Metrics Only

In [ ]:
# Key metrics!
focus_cols = ["hr", "r", "rbi", "slg", "weighted_runs_created_plus", "ops", "iso",
              "barrel_percentage", "hard_hit_percentage", "avg_exit_velocity",
              "avg_launch_angle", "max_exit_velocity", "expected_weighted_on_base_average",
              "bb_percentage", "so_percentage", "walks_to_strikeouts_ratio"]

# Compute correlation matrix!
corr_focus = all[focus_cols].corr()

# Plot heatmap!
plt.figure(figsize = FIG)
sns.heatmap(corr_focus,
            cmap = "coolwarm",
            center = 0,
            square = False,
            linewidths = 0.1)
plt.title("Correlation Heatmap For Key Metrics", fontsize = FS + 8)
plt.tight_layout()
plt.show()

Note: 

- The full correlation heatmap for all Statcast and traditional hitting metrics reveals that MLB offensive performance is organized into several coherent statistical clusters rather than a collection of independent variables. Counting metrics such as games, PA, H, HR, and RBI form the first major block of strong positive correlations. This reflects the fact that these measures are primarily driven by opportunity rather than underlying skill: *players who accumulate more playing time naturally accrue more counting statistics*. A second cluster centers on plate discipline and contact-oriented metrics, including BB percentage, SO percentage, swing tendencies, and contact percentages, which display predictable relationships. For example, contact percentage and swinging-strike percentage are strongly negatively correlated, while out-of-zone swing percentage correlates negatively with walk percentage. This structure captures the league-wide tradeoff between discipline/swing decisions, and contact quality.

- The heatmap focusing on key performance shows that power-oriented outcome metrics such as HR, SLG, ISO, OPS, and wRC+ form an exceptionally tight cluster, as expected, reflecting that they are different expressions of the same underlying dimension of offensive production. Contact quality measures, especially barrel percentage and hard-hit percentage, are among the strongest league-wide predictors of these outcomes. LA shows a positive but noticeably weaker correlation with power metrics, suggesting that while batted-ball trajectory modulates offensive output, it is secondary to the more fundamental effect of contact quality. *Plate discipline metrics* also show meaningful league-wide relationships: BB percentage contributes positively to wRC+ and OPS, while SO percentage is negatively associated with virtually all measures of offensive value, again, as one would expect. Together, these results illustrate that, across MLB hitters, the most important drivers of run production are a combination of high-quality contact and sound plate discipline.

## 9. Making Pairplots (Judge Again Highlighted By Season In Red)

In [ ]:
# Subsets of metrics!
contact_quality = ["avg_exit_velocity", 
                   "max_exit_velocity",
                   "hard_hit_percentage",
                   "sweet_spot_percentage", 
                   "barrel_percentage", 
                   "line_drive_percentage"]
plate_discipline = ["bb_percentage",
                    "so_percentage",
                    "walks_to_strikeouts_ratio",
                    "outside_swing_percentage",
                    "foul_strike_percentage",
                    "swinging_strike_percentage"]
outcomes = ["hr", "slg", "ops", "iso", "weighted_runs_created_plus", "wins_above_replacement"]

# Add Judge flag!
all_copy = all.copy()
all_copy["is_judge"] = all_copy["mlbam_id"] == 592450

# Pretty labels (no raw df names on axes)!
label_map = {"avg_exit_velocity": "Average EV (in mph)",
             "max_exit_velocity": "Maximum EV (in mph)",
             "hard_hit_percentage": "Hard Hit Percentage",
             "sweet_spot_percentage": "Sweet Spot Percentage",
             "barrel_percentage": "Barrel Percentage",
             "line_drive_percentage": "Line Drive Percentage",
             "bb_percentage": "BB Percentage",
             "so_percentage": "SO Percentage",
             "walks_to_strikeouts_ratio": "Walk-to-Strikeout Ratio",
             "outside_swing_percentage": "Outside Swing (Chase) Percentage",
             "foul_strike_percentage": "Foul Strike Percentage",
             "swinging_strike_percentage": "Swinging Strike Percentage",
             "hr": "HR",
             "slg": "SLG",
             "ops": "OPS",
             "iso": "ISO",
             "weighted_runs_created_plus": "wRC+",
             "wins_above_replacement": "WAR"}

# Make a function to plot!
def cross_scatter_grid(df, row_vars, col_vars, title):
    n_rows = len(row_vars)
    n_cols = len(col_vars)
    fig, axes = plt.subplots(n_rows, 
                             n_cols,
                             figsize = (4 * n_cols, 3.2 * n_rows),
                             squeeze = False)
    for i, ycol in enumerate(row_vars):
        for j, xcol in enumerate(col_vars):
            ax = axes[i, j]
            sns.scatterplot(data = df,
                            x = xcol,
                            y = ycol,
                            hue = "is_judge",
                            palette = {False: "gray", True: "red"},
                            alpha = 0.4,
                            s = 20,
                            legend = False,
                            ax = ax)
            ax.set_xlabel(label_map[xcol], fontsize = FS - 4)
            ax.set_ylabel(label_map[ycol], fontsize = FS - 4)
    fig.suptitle(title, fontsize = FS, y = 0.995)
    plt.tight_layout(rect = (0, 0, 1, 0.97))
    plt.show()

# Call the function!
cross_scatter_grid(all_copy,
                   row_vars = outcomes,
                   col_vars = contact_quality,
                   title = "Contact Quality Metrics vs. Power/Run Value Outcomes")
cross_scatter_grid(all_copy,
                   row_vars = outcomes,
                   col_vars = plate_discipline,
                   title = "Plate Discipline Metrics vs. Power/Run Value Outcomes")

Note: 

- To visualize the league-wide relationships suggested by the correlation heatmaps, I constructed grids of scatter plots pairing contact quality metrics with power and run value outcomes for every qualified MLB hitter in the sample. Across all combinations, the pattern is monotone: *players who hit the ball harder and on more optimal trajectories systematically generate more HR, higher SLG, ISO (just examples), and overall offensive value*. The tightest relationship arises for barrel percentage with isolated power, which is intuitive. Very few seasons with exceptional contact quality measures produce merely average power output, and almost no high-wRC+ or high-WAR seasons occur with low values of these contact quality measures. Average and maximum EV, hard hit percentage, and sweet spot percentage also show clear positive trends, though with more dispersion.

- The second grid relates plate discipline metrics to the same outcome variables. BB percentage is positively associated with wRC+, OPS, and WAR, and SO percentage and swinging-strike percentage are negatively associated with these measures, as one would expect, *but, with not just BB percentage, the corresponding clouds are noticeably wider than in the contact quality plots*. Offensive value clearly benefits from drawing walks and avoiding strikeouts, yet there exist many high-strikeout, high-power players as well as low-walk hitters who remain productive due to exceptional contact quality. In contrast, swing tendencies exhibit only weak league-wide relationships with power and run value. Taken together, these pair plots show that contact quality correlates more statistically with power and run value outcomes than plate discipline. However, good plate discipline is also important for producing runs, and is considered colloquially to be the precursor to contact quality. Therefore, hitters that produce as many runs as Aaron Judge must have a combination of high contact quality and discipline at the plate (lots of BBs, not chasing pitches outside of the strike zone, etc.). 